## Motor Vehicle Collisions Data Cleaning

### 1.Initial the Collisions Data

In [1]:
import pandas as pd
import numpy as np
import pymysql
from sqlalchemy import create_engine

In [4]:
df = pd.read_csv(r'D:\Data Analysis\1.Projects\2.Motor Vehicle Collisions Analysis\Resources\Motor_Vehicle_Collisions_Crashes.csv')

C:\Users\徐海心\AppData\Local\Temp\ipykernel_21764\340890899.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'D:\Data Analysis\1.Projects\2.Motor Vehicle Collisions Analysis\Resources\Motor_Vehicle_Collisions_Crashes.csv')


In [84]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2206393 entries, 0 to 2206392
Data columns (total 29 columns):
 #   Column                         Dtype  
---  ------                         -----  
 0   CRASH DATE                     object 
 1   CRASH TIME                     object 
 2   BOROUGH                        object 
 3   ZIP CODE                       object 
 4   LATITUDE                       float64
 5   LONGITUDE                      float64
 6   LOCATION                       object 
 7   ON STREET NAME                 object 
 8   CROSS STREET NAME              object 
 9   OFF STREET NAME                object 
 10  NUMBER OF PERSONS INJURED      float64
 11  NUMBER OF PERSONS KILLED       float64
 12  NUMBER OF PEDESTRIANS INJURED  int64  
 13  NUMBER OF PEDESTRIANS KILLED   int64  
 14  NUMBER OF CYCLIST INJURED      int64  
 15  NUMBER OF CYCLIST KILLED       int64  
 16  NUMBER OF MOTORIST INJURED     int64  
 17  NUMBER OF MOTORIST KILLED      int64  
 18  CO

In [45]:
pd.set_option('display.max.rows', 2000)
pd.set_option('display.max.columns', 50)

In [5]:
df.columns = (df.columns
              .str.lower()
              .str.strip()
              .str.replace(' ','_')
             )

In [6]:
df.head()
# the "on street" is your main road, the "cross street" is the intersecting road, and an "off street" refers to a smaller road 

,crash_date,crash_time,borough,zip_code,latitude,longitude,location,on_street_name,cross_street_name,off_street_name,...,contributing_factor_vehicle_2,contributing_factor_vehicle_3,contributing_factor_vehicle_4,contributing_factor_vehicle_5,collision_id,vehicle_type_code_1,vehicle_type_code_2,vehicle_type_code_3,vehicle_type_code_4,vehicle_type_code_5
0,09/11/2021,2:39,NaN,NaN,NaN,NaN,NaN,WHITESTONE EXPRESSWAY,20 AVENUE,NaN,...,Unspecified,NaN,NaN,NaN,4455765,Sedan,Sedan,NaN,NaN,NaN
1,03/26/2022,11:45,NaN,NaN,NaN,NaN,NaN,QUEENSBORO BRIDGE UPPER,NaN,NaN,...,NaN,NaN,NaN,NaN,4513547,Sedan,NaN,NaN,NaN,NaN
2,11/01/2023,1:29,BROOKLYN,11230.0,40.62179,-73.970024,"(40.62179, -73.970024)",OCEAN PARKWAY,AVENUE K,NaN,...,Unspecified,Unspecified,NaN,NaN,4675373,Moped,Sedan,Sedan,NaN,NaN
3,06/29/2022,6:55,NaN,NaN,NaN,NaN,NaN,THROGS NECK BRIDGE,NaN,NaN,...,Unspecified,NaN,NaN,NaN,4541903,Sedan,Pick-up Truck,NaN,NaN,NaN
4,09/21/2022,13:21,NaN,NaN,NaN,NaN,NaN,BROOKLYN BRIDGE,NaN,NaN,...,Unspecified,NaN,NaN,NaN,4566131,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN,NaN


### 2. Drop the Duplicates

In [87]:
df.duplicated(subset = 'collision_id').sum()

np.int64(0)

In [88]:
(df.duplicated(subset = ['crash_date','crash_time','location'],keep = False) & ~df['location'].isna()).sum()

np.int64(11662)

In [7]:
duplicates = df[df.duplicated(subset = ['crash_date','crash_time','location'],keep = False) & df['location'].isna()]

In [90]:
duplicates

,crash_date,crash_time,borough,zip_code,latitude,longitude,location,on_street_name,cross_street_name,off_street_name,number_of_persons_injured,number_of_persons_killed,number_of_pedestrians_injured,number_of_pedestrians_killed,number_of_cyclist_injured,number_of_cyclist_killed,number_of_motorist_injured,number_of_motorist_killed,contributing_factor_vehicle_1,contributing_factor_vehicle_2,contributing_factor_vehicle_3,contributing_factor_vehicle_4,contributing_factor_vehicle_5,collision_id,vehicle_type_code_1,vehicle_type_code_2,vehicle_type_code_3,vehicle_type_code_4,vehicle_type_code_5
1,03/26/2022,11:45,NaN,NaN,NaN,NaN,NaN,QUEENSBORO BRIDGE UPPER,NaN,NaN,1.0,0.0,0,0,0,0,1,0,Pavement Slippery,NaN,NaN,NaN,NaN,4513547,Sedan,NaN,NaN,NaN,NaN
18,12/14/2021,8:30,NaN,NaN,NaN,NaN,NaN,broadway,west 80 street -west 81 street,NaN,0.0,0.0,0,0,0,0,0,0,Unsafe Lane Changing,Unspecified,NaN,NaN,NaN,4486634,Station Wagon/Sport Utility Vehicle,Sedan,NaN,NaN,NaN
160,04/13/2021,16:00,BROOKLYN,11222.0,NaN,NaN,NaN,VANDERVORT AVENUE,ANTHONY STREET,NaN,0.0,0.0,0,0,0,0,0,0,Following Too Closely,Unspecified,NaN,NaN,NaN,4407811,Sedan,NaN,NaN,NaN,NaN
245,09/11/2021,15:25,NaN,NaN,NaN,NaN,NaN,VERRAZANO BRIDGE UPPER,NaN,NaN,0.0,0.0,0,0,0,0,0,0,Unspecified,Unspecified,NaN,NaN,NaN,4456122,Sedan,Sedan,NaN,NaN,NaN
293,09/11/2021,14:00,NaN,NaN,NaN,NaN,NaN,CROSS BRONX EXPY RAMP,NaN,NaN,2.0,0.0,0,0,0,0,2,0,Following Too Closely,Unspecified,NaN,NaN,NaN,4456497,Station Wagon/Sport Utility Vehicle,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2159871,03/03/2025,11:00,BROOKLYN,11224.0,NaN,NaN,NaN,WEST 27 STREET,NEPTUNE AVENUE,NaN,1.0,0.0,0,0,1,0,0,0,Unspecified,Unspecified,NaN,NaN,NaN,4796092,Station Wagon/Sport Utility Vehicle,Bike,NaN,NaN,NaN
2163406,03/20/2025,8:30,BROOKLYN,11216.0,NaN,NaN,NaN,BROOKLYN AVE,FULTON ST,NaN,0.0,0.0,0,0,0,0,0,0,Backing Unsafely,Unspecified,NaN,NaN,NaN,4800122,Sedan,Sedan,NaN,NaN,NaN
2164913,03/20/2025,8:30,NaN,NaN,NaN,NaN,NaN,ATLANTIC AVE,Pennsylvania,NaN,0.0,0.0,0,0,0,0,0,0,Unsafe Lane Changing,Unspecified,Unspecified,NaN,NaN,4801507,Station Wagon/Sport Utility Vehicle,Station Wagon/Sport Utility Vehicle,Sedan,NaN,NaN
2169744,04/17/2025,1:19,MANHATTAN,10009.0,NaN,NaN,NaN,14 st,1 ave,NaN,2.0,0.0,0,0,0,0,2,0,Driver Inattention/Distraction,Turning Improperly,NaN,NaN,NaN,4806319,Bus,NaN,NaN,NaN,NaN


In [91]:
df['crash_date'].isna().sum()

np.int64(0)

In [92]:
df['crash_time'].isna().sum()

np.int64(0)

In [93]:
df['location'].isna().sum()

np.int64(240227)

In [8]:
# The collusions which happened at the identical time and location should be indentified as the same collision, even thongh the collusion ids are different.
# First make a mask with nan location duplicates
duplicates_nan_location = df.duplicated(subset = ['crash_date','crash_time','location'],keep = False) & df['location'].isna()

In [9]:
df_not_nan_duplicates = df[~duplicates_nan_location]

In [10]:
df_not_nan_duplicates[df_not_nan_duplicates.duplicated(subset = ['crash_date','crash_time','location'],keep = False)]

,crash_date,crash_time,borough,zip_code,latitude,longitude,location,on_street_name,cross_street_name,off_street_name,...,contributing_factor_vehicle_2,contributing_factor_vehicle_3,contributing_factor_vehicle_4,contributing_factor_vehicle_5,collision_id,vehicle_type_code_1,vehicle_type_code_2,vehicle_type_code_3,vehicle_type_code_4,vehicle_type_code_5
565,04/16/2021,7:45,QUEENS,11377.0,40.733597,-73.910620,"(40.733597, -73.91062)",NaN,NaN,53-15 58 STREET,...,Unspecified,NaN,NaN,NaN,4407834,Sedan,NaN,NaN,NaN,NaN
631,04/16/2021,0:30,QUEENS,11369.0,40.758680,-73.875520,"(40.75868, -73.87552)",93 STREET,32 AVENUE,NaN,...,Unspecified,Unspecified,NaN,NaN,4407714,Sedan,Bike,NaN,NaN,NaN
787,04/16/2021,0:30,QUEENS,11369.0,40.758680,-73.875520,"(40.75868, -73.87552)",93 STREET,32 AVENUE,NaN,...,Unspecified,NaN,NaN,NaN,4407713,Sedan,NaN,NaN,NaN,NaN
1512,03/25/2021,20:00,QUEENS,11373.0,0.000000,0.000000,"(0.0, 0.0)",82 STREET,GRAND AVENUE,NaN,...,Reaction to Uninvolved Vehicle,Unspecified,Unspecified,NaN,4408756,Sedan,Sedan,Sedan,bus,NaN
1718,04/17/2021,11:40,QUEENS,11411.0,40.696373,-73.745250,"(40.696373, -73.74525)",207 STREET,LINDEN BOULEVARD,NaN,...,Unspecified,NaN,NaN,NaN,4407940,Sedan,FDNY ENGIN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2203235,08/26/2025,9:30,MANHATTAN,10004.0,40.706700,-74.016045,"(40.7067, -74.016045)",WEST ST,MORRIS ST,NaN,...,Unspecified,NaN,NaN,NaN,4839942,Station Wagon/Sport Utility Vehicle,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN
2203564,08/13/2025,10:25,QUEENS,11434.0,40.673008,-73.788086,"(40.673008, -73.788086)",150 ST,ROCKAWAY BLVD,NaN,...,NaN,NaN,NaN,NaN,4840192,Standing S,NaN,NaN,NaN,NaN
2203963,09/06/2025,17:45,BRONX,10473.0,40.823600,-73.870040,"(40.8236, -73.87004)",NaN,NaN,928 FTELEY AVE,...,Other Vehicular,NaN,NaN,NaN,4840562,Sedan,Sedan,NaN,NaN,NaN
2204473,07/31/2025,16:47,QUEENS,11372.0,40.756340,-73.877930,"(40.75634, -73.87793)",90 ST,NORTHERN BLVD,NaN,...,Unspecified,NaN,NaN,NaN,4841747,Sedan,Moped,NaN,NaN,NaN


In [11]:
df_drop_duplicates = df_not_nan_duplicates.drop_duplicates(subset = ['crash_date','crash_time','location'],keep = 'first')

In [10]:
df_drop_duplicates.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2140088 entries, 0 to 2206392
Data columns (total 29 columns):
 #   Column                         Dtype  
---  ------                         -----  
 0   crash_date                     object 
 1   crash_time                     object 
 2   borough                        object 
 3   zip_code                       object 
 4   latitude                       float64
 5   longitude                      float64
 6   location                       object 
 7   on_street_name                 object 
 8   cross_street_name              object 
 9   off_street_name                object 
 10  number_of_persons_injured      float64
 11  number_of_persons_killed       float64
 12  number_of_pedestrians_injured  int64  
 13  number_of_pedestrians_killed   int64  
 14  number_of_cyclist_injured      int64  
 15  number_of_cyclist_killed       int64  
 16  number_of_motorist_injured     int64  
 17  number_of_motorist_killed      int64  
 18  contrib

In [12]:
df = pd.concat([df_drop_duplicates,duplicates])

In [13]:
df[df.duplicated(subset = ['crash_date','crash_time','location'],keep = False) & ~df['location'].isna()]

,crash_date,crash_time,borough,zip_code,latitude,longitude,location,on_street_name,cross_street_name,off_street_name,...,contributing_factor_vehicle_2,contributing_factor_vehicle_3,contributing_factor_vehicle_4,contributing_factor_vehicle_5,collision_id,vehicle_type_code_1,vehicle_type_code_2,vehicle_type_code_3,vehicle_type_code_4,vehicle_type_code_5


### 3. Type Conversion

In [14]:
df['crash_datetime'] = df['crash_date'] + df['crash_time']

In [15]:
df['crash_datetime'] = pd.to_datetime(df['crash_datetime'],format = '%m/%d/%Y%H:%M')

In [16]:
crash_datetime = df.pop('crash_datetime')
df.insert(2,'crash_datetime',crash_datetime)

In [17]:
df.head()

,crash_date,crash_time,crash_datetime,borough,zip_code,latitude,longitude,location,on_street_name,cross_street_name,...,contributing_factor_vehicle_2,contributing_factor_vehicle_3,contributing_factor_vehicle_4,contributing_factor_vehicle_5,collision_id,vehicle_type_code_1,vehicle_type_code_2,vehicle_type_code_3,vehicle_type_code_4,vehicle_type_code_5
0,09/11/2021,2:39,2021-09-11 02:39:00,NaN,NaN,NaN,NaN,NaN,WHITESTONE EXPRESSWAY,20 AVENUE,...,Unspecified,NaN,NaN,NaN,4455765,Sedan,Sedan,NaN,NaN,NaN
2,11/01/2023,1:29,2023-11-01 01:29:00,BROOKLYN,11230.0,40.62179,-73.970024,"(40.62179, -73.970024)",OCEAN PARKWAY,AVENUE K,...,Unspecified,Unspecified,NaN,NaN,4675373,Moped,Sedan,Sedan,NaN,NaN
3,06/29/2022,6:55,2022-06-29 06:55:00,NaN,NaN,NaN,NaN,NaN,THROGS NECK BRIDGE,NaN,...,Unspecified,NaN,NaN,NaN,4541903,Sedan,Pick-up Truck,NaN,NaN,NaN
4,09/21/2022,13:21,2022-09-21 13:21:00,NaN,NaN,NaN,NaN,NaN,BROOKLYN BRIDGE,NaN,...,Unspecified,NaN,NaN,NaN,4566131,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN,NaN
5,04/26/2023,13:30,2023-04-26 13:30:00,NaN,NaN,NaN,NaN,NaN,WEST 54 STREET,NaN,...,Unspecified,NaN,NaN,NaN,4623759,Sedan,Box Truck,NaN,NaN,NaN


### 4. Populating the Missing Values (boroughs,streets,locations)

In [18]:
df['on_street_name'] = df['on_street_name'].str.strip()
df['on_street_name'] = df['on_street_name'].replace('',np.nan)

In [84]:
dict_borough_street = borough_on_street.set_index('on_street_name')['borough'].to_dict()

In [20]:
(df['borough'].isna() & ~df['on_street_name'].isna()).sum() 

np.int64(547727)

In [21]:
df['borough'] = df['borough'].fillna(df.groupby('on_street_name')['borough'].transform('first'))

In [22]:
df['borough'] = df['borough'].fillna(df.groupby('location')['borough'].transform('first'))

In [23]:
df['zip_code'] = df['zip_code'].fillna(df.groupby('on_street_name')['zip_code'].transform('first'))

In [24]:
df['zip_code'] = df['zip_code'].fillna(df.groupby('location')['zip_code'].transform('first'))

In [33]:
df['borough'].unique()

array(['QUEENS', 'BROOKLYN', nan, 'MANHATTAN', 'BRONX', 'STATEN ISLAND',
       None], dtype=object)

In [38]:
df.isna().sum()

crash_date                             0
crash_time                             0
crash_datetime                         0
borough                           143748
zip_code                          143820
latitude                          240227
longitude                         240227
location                          240227
on_street_name                    478031
cross_street_name                 840313
off_street_name                  1814151
number_of_persons_injured             16
number_of_persons_killed              27
number_of_pedestrians_injured          0
number_of_pedestrians_killed           0
number_of_cyclist_injured              0
number_of_cyclist_killed               0
number_of_motorist_injured             0
number_of_motorist_killed              0
contributing_factor_vehicle_1       7701
contributing_factor_vehicle_2     351800
contributing_factor_vehicle_3    2041355
contributing_factor_vehicle_4    2164129
contributing_factor_vehicle_5    2190482
collision_id    

### 5. Standarlising the Vehicle Type

In [77]:
vehicle = df[['vehicle_type_code_1','vehicle_type_code_2','vehicle_type_code_3','vehicle_type_code_4','vehicle_type_code_5']]

In [78]:
vehicle_500 = vehicle[:500]

In [68]:
vehicle_500.info()

<class 'pandas.core.frame.DataFrame'>
Index: 500 entries, 0 to 506
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   vehicle_type_code_1  492 non-null    object
 1   vehicle_type_code_2  326 non-null    object
 2   vehicle_type_code_3  41 non-null     object
 3   vehicle_type_code_4  9 non-null      object
 4   vehicle_type_code_5  2 non-null      object
dtypes: object(5)
memory usage: 23.4+ KB


In [81]:
vehicle_500['type_1'] = vehicle_500['vehicle_type_code_1'].str.extract(r'^([^/]*)', expand=False)

C:\Users\徐海心\AppData\Local\Temp\ipykernel_2452\3631606616.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  vehicle_500['type_1'] = vehicle_500['vehicle_type_code_1'].str.extract(r'^([^/]*)', expand=False)


In [34]:
df['vehicle_type_code_1'] = df['vehicle_type_code_1'].str.extract(r'^([^/]*)', expand=False)

In [35]:
df['vehicle_type_code_2'] = df['vehicle_type_code_2'].str.extract(r'^([^/]*)', expand=False)

In [36]:
df[['vehicle_type_code_1','vehicle_type_code_2','vehicle_type_code_3','vehicle_type_code_4','vehicle_type_code_5']]

,vehicle_type_code_1,vehicle_type_code_2,vehicle_type_code_3,vehicle_type_code_4,vehicle_type_code_5
0,Sedan,Sedan,NaN,NaN,NaN
2,Moped,Sedan,Sedan,NaN,NaN
3,Sedan,Pick-up Truck,NaN,NaN,NaN
4,Station Wagon,NaN,NaN,NaN,NaN
5,Sedan,Box Truck,NaN,NaN,NaN
...,...,...,...,...,...
2159871,Station Wagon,Bike,NaN,NaN,NaN
2163406,Sedan,Sedan,NaN,NaN,NaN
2164913,Station Wagon,Station Wagon,Sedan,NaN,NaN
2169744,Bus,NaN,NaN,NaN,NaN


In [88]:
df['vehicle_type_code_1'].nunique()

1796

### 6. Remove Irrelevant Columns

In [89]:
df.columns

Index(['crash_date', 'crash_time', 'crash_datetime', 'borough', 'zip_code',
       'latitude', 'longitude', 'location', 'on_street_name',
       'cross_street_name', 'off_street_name', 'number_of_persons_injured',
       'number_of_persons_killed', 'number_of_pedestrians_injured',
       'number_of_pedestrians_killed', 'number_of_cyclist_injured',
       'number_of_cyclist_killed', 'number_of_motorist_injured',
       'number_of_motorist_killed', 'contributing_factor_vehicle_1',
       'contributing_factor_vehicle_2', 'contributing_factor_vehicle_3',
       'contributing_factor_vehicle_4', 'contributing_factor_vehicle_5',
       'collision_id', 'vehicle_type_code_1', 'vehicle_type_code_2',
       'vehicle_type_code_3', 'vehicle_type_code_4', 'vehicle_type_code_5'],
      dtype='object')

In [37]:
df = df.drop(['crash_date','crash_time','latitude', 'longitude', 'location', 'off_street_name'],axis = 1)

In [38]:
df.head()

,crash_datetime,borough,zip_code,on_street_name,cross_street_name,number_of_persons_injured,number_of_persons_killed,number_of_pedestrians_injured,number_of_pedestrians_killed,number_of_cyclist_injured,...,contributing_factor_vehicle_2,contributing_factor_vehicle_3,contributing_factor_vehicle_4,contributing_factor_vehicle_5,collision_id,vehicle_type_code_1,vehicle_type_code_2,vehicle_type_code_3,vehicle_type_code_4,vehicle_type_code_5
0,2021-09-11 02:39:00,QUEENS,11354.0,WHITESTONE EXPRESSWAY,20 AVENUE,2.0,0.0,0,0,0,...,Unspecified,NaN,NaN,NaN,4455765,Sedan,Sedan,NaN,NaN,NaN
2,2023-11-01 01:29:00,BROOKLYN,11230.0,OCEAN PARKWAY,AVENUE K,1.0,0.0,0,0,0,...,Unspecified,Unspecified,NaN,NaN,4675373,Moped,Sedan,Sedan,NaN,NaN
3,2022-06-29 06:55:00,NaN,NaN,THROGS NECK BRIDGE,NaN,0.0,0.0,0,0,0,...,Unspecified,NaN,NaN,NaN,4541903,Sedan,Pick-up Truck,NaN,NaN,NaN
4,2022-09-21 13:21:00,BROOKLYN,11201.0,BROOKLYN BRIDGE,NaN,0.0,0.0,0,0,0,...,Unspecified,NaN,NaN,NaN,4566131,Station Wagon,NaN,NaN,NaN,NaN
5,2023-04-26 13:30:00,MANHATTAN,10019.0,WEST 54 STREET,NaN,0.0,0.0,0,0,0,...,Unspecified,NaN,NaN,NaN,4623759,Sedan,Box Truck,NaN,NaN,NaN


### 7. Formating the Values

In [57]:
df['borough'] = df['borough'].str.title()
df['on_street_name'] = df['on_street_name'].str.title()
df['cross_street_name'] = df['cross_street_name'].str.title()

### 8. Working With the Null Values

In [39]:
df.isna().sum()

crash_datetime                         0
borough                           143748
zip_code                          143820
on_street_name                    478031
cross_street_name                 840313
number_of_persons_injured             16
number_of_persons_killed              27
number_of_pedestrians_injured          0
number_of_pedestrians_killed           0
number_of_cyclist_injured              0
number_of_cyclist_killed               0
number_of_motorist_injured             0
number_of_motorist_killed              0
contributing_factor_vehicle_1       7701
contributing_factor_vehicle_2     351800
contributing_factor_vehicle_3    2041355
contributing_factor_vehicle_4    2164129
contributing_factor_vehicle_5    2190482
collision_id                           0
vehicle_type_code_1                15742
vehicle_type_code_2               439169
vehicle_type_code_3              2047507
vehicle_type_code_4              2165441
vehicle_type_code_5              2190792
dtype: int64

In [40]:
df['borough'] = df['borough'].fillna('Unspecified')
df['zip_code'] = df['zip_code'].fillna('Unspecified')
df['number_of_persons_injured'] = df['number_of_persons_injured'].fillna('Not Counted')
df['number_of_persons_killed'] = df['number_of_persons_killed'].fillna('Not Counted')

df['on_street_name'] = df['on_street_name'].fillna('Unspecified')
df['cross_street_name'] = df['cross_street_name'].fillna('Unspecified')

df['contributing_factor_vehicle_1'] = df['contributing_factor_vehicle_1'].fillna('Unspecified')
df['contributing_factor_vehicle_2'] = df['contributing_factor_vehicle_2'].fillna('Unspecified')
df['contributing_factor_vehicle_3'] = df['contributing_factor_vehicle_3'].fillna('Unspecified')
df['contributing_factor_vehicle_4'] = df['contributing_factor_vehicle_4'].fillna('Unspecified')
df['contributing_factor_vehicle_5'] = df['contributing_factor_vehicle_5'].fillna('Unspecified')

df['vehicle_type_code_1'] = df['vehicle_type_code_1'].fillna('Unspecified')
df['vehicle_type_code_2'] = df['vehicle_type_code_2'].fillna('Unspecified')
df['vehicle_type_code_3'] = df['vehicle_type_code_3'].fillna('Unspecified')
df['vehicle_type_code_4'] = df['vehicle_type_code_4'].fillna('Unspecified')
df['vehicle_type_code_5'] = df['vehicle_type_code_5'].fillna('Unspecified')



In [41]:
df.isna().sum()

crash_datetime                   0
borough                          0
zip_code                         0
on_street_name                   0
cross_street_name                0
number_of_persons_injured        0
number_of_persons_killed         0
number_of_pedestrians_injured    0
number_of_pedestrians_killed     0
number_of_cyclist_injured        0
number_of_cyclist_killed         0
number_of_motorist_injured       0
number_of_motorist_killed        0
contributing_factor_vehicle_1    0
contributing_factor_vehicle_2    0
contributing_factor_vehicle_3    0
contributing_factor_vehicle_4    0
contributing_factor_vehicle_5    0
collision_id                     0
vehicle_type_code_1              0
vehicle_type_code_2              0
vehicle_type_code_3              0
vehicle_type_code_4              0
vehicle_type_code_5              0
dtype: int64

In [42]:
df['borough'].unique()

array(['QUEENS', 'BROOKLYN', 'Unspecified', 'MANHATTAN', 'BRONX',
       'STATEN ISLAND'], dtype=object)

In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2200418 entries, 0 to 2171470
Data columns (total 24 columns):
 #   Column                         Dtype         
---  ------                         -----         
 0   crash_datetime                 datetime64[ns]
 1   borough                        object        
 2   zip_code                       object        
 3   on_street_name                 object        
 4   cross_street_name              object        
 5   number_of_persons_injured      object        
 6   number_of_persons_killed       object        
 7   number_of_pedestrians_injured  int64         
 8   number_of_pedestrians_killed   int64         
 9   number_of_cyclist_injured      int64         
 10  number_of_cyclist_killed       int64         
 11  number_of_motorist_injured     int64         
 12  number_of_motorist_killed      int64         
 13  contributing_factor_vehicle_1  object        
 14  contributing_factor_vehicle_2  object        
 15  contributing_factor_

In [59]:
df.head()

,crash_datetime,borough,zip_code,on_street_name,cross_street_name,number_of_persons_injured,number_of_persons_killed,number_of_pedestrians_injured,number_of_pedestrians_killed,number_of_cyclist_injured,number_of_cyclist_killed,number_of_motorist_injured,number_of_motorist_killed,contributing_factor_vehicle_1,contributing_factor_vehicle_2,contributing_factor_vehicle_3,contributing_factor_vehicle_4,contributing_factor_vehicle_5,collision_id,vehicle_type_code_1,vehicle_type_code_2,vehicle_type_code_3,vehicle_type_code_4,vehicle_type_code_5
0,2021-09-11 02:39:00,Queens,11354.0,Whitestone Expressway,20 Avenue,2.0,0.0,0,0,0,0,2,0,Aggressive Driving/Road Rage,Unspecified,Unspecified,Unspecified,Unspecified,4455765,Sedan,Sedan,Unspecified,Unspecified,Unspecified
2,2023-11-01 01:29:00,Brooklyn,11230.0,Ocean Parkway,Avenue K,1.0,0.0,0,0,0,0,1,0,Unspecified,Unspecified,Unspecified,Unspecified,Unspecified,4675373,Moped,Sedan,Sedan,Unspecified,Unspecified
3,2022-06-29 06:55:00,Unspecified,Unspecified,Throgs Neck Bridge,Unspecified,0.0,0.0,0,0,0,0,0,0,Following Too Closely,Unspecified,Unspecified,Unspecified,Unspecified,4541903,Sedan,Pick-up Truck,Unspecified,Unspecified,Unspecified
4,2022-09-21 13:21:00,Brooklyn,11201.0,Brooklyn Bridge,Unspecified,0.0,0.0,0,0,0,0,0,0,Passing Too Closely,Unspecified,Unspecified,Unspecified,Unspecified,4566131,Station Wagon,Unspecified,Unspecified,Unspecified,Unspecified
5,2023-04-26 13:30:00,Manhattan,10019.0,West 54 Street,Unspecified,0.0,0.0,0,0,0,0,0,0,Unspecified,Unspecified,Unspecified,Unspecified,Unspecified,4623759,Sedan,Box Truck,Unspecified,Unspecified,Unspecified


### 9. Export the data to mysql for further analysis

In [60]:
# Preparing the grounds for export
username = "root"
password = "Sanchahe851225"
host = "localhost"
port = "3306"
database = "motor_vehicle_collisions"

# Create connection engine
engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")


In [ ]:
# Export to sql
df.to_sql(
    name="cleaned_collisions_data_backup",   # Table name
    con=engine,
    if_exists="replace",              # or "append" to add without deleting existing
    index=False,                      # Avoid exporting pandas index
    chunksize=1000                    # Handle large datasets in batches
)

In [62]:
# Export clean data to csv for later use
df.to_csv("cleaned_collisions_data.csv", index = False)